# SVM Model 2: RSB Kernel
David Beas, Nick Garcia

https://archive.ics.uci.edu/dataset/597/garments+worker+productivity

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Load the dataset
df = pd.read_csv('garments_worker_productivity.csv')

# Data Cleaning: Handling missing values and unnecessary strings
# Drop the 'date' column as specific string dates do not process well in SVM math
df = df.drop(columns=['date'])

# The 'wip' (Work in Progress) column has many blanks. In garment manufacturing,
# 'finishing' departments often have no WIP. We fill the blanks with 0.
df['wip'] = df['wip'].fillna(0)

# Clean up hidden spaces in the text columns
df['department'] = df['department'].str.strip()

# Formulate the Target Variable for Classification
# We want to predict if a team will successfully meet their targeted productivity.
# If actual >= targeted, it's a 1 (Success). If actual < targeted, it's a 0 (Fail).
df['met_target'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)

# Encode Categorical Variables into numbers
le_quarter = LabelEncoder()
df['quarter'] = le_quarter.fit_transform(df['quarter'])

le_dept = LabelEncoder()
df['department'] = le_dept.fit_transform(df['department'])

le_day = LabelEncoder()
df['day'] = le_day.fit_transform(df['day'])

df.head()

In [ ]:
# Define Features and Target Variables
# We drop 'actual_productivity' so the model doesn't cheat, but we keep 'targeted_productivity'
# as a feature to see if the goal itself was realistic given the team size and resources.
X = df.drop(columns=['actual_productivity', 'met_target'])
y = df['met_target']

# Split the data into Training (75%) and Testing (25%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Feature Scaling
# The RSB kernel measures the geometric distance between data points. We must scale
# because large numbers (Overtime = 7000) will overpower small ones (Workers = 50).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train the SVM using the RBF Kernel
# We use gamma='scale' and C=1.0 as standard reliable parameters for RBF boundaries
svm_rbf = SVC(kernel='rbf', gamma='scale', C=1.0, random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

# Make predictions
y_pred = svm_rbf.predict(X_test_scaled)

# Print Reliability Metrics
print("--- Support Vector Machine: RBF Kernel ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")

In [ ]:
# User Input Provision for New Predictions
def predict_productivity():
    while True:
        try:
            print("\n--- Factory Team Target Predictor ---")
            # Get user inputs
            quarter = int(input("Enter Quarter (0=Q1, 1=Q2, 2=Q3, 3=Q4, 4=Q5): "))
            dept = int(input("Enter Department (0=Finishing, 1=Sewing): "))
            day = int(input("Enter Day (0=Mon, 1=Sat, 2=Sun, 3=Thu, 4=Tue, 5=Wed): "))
            team = int(input("Enter Team Number (1-12): "))
            target = float(input("Enter Targeted Productivity (e.g., 0.80): "))
            smv = float(input("Enter SMV / Standard Minute Value (e.g., 22.5): "))
            wip = int(input("Enter WIP / Work In Progress items (0 if none): "))
            overtime = int(input("Enter Overtime Minutes (e.g., 5000): "))
            incentive = int(input("Enter Financial Incentive (e.g., 50): "))
            idle_time = float(input("Enter Idle Time Minutes (e.g., 0): "))
            idle_men = int(input("Enter Number of Idle Men (e.g., 0): "))
            style_changes = int(input("Enter Number of Style Changes (e.g., 0 or 1): "))
            workers = float(input("Enter Total Number of Workers (e.g., 55): "))

            # Create dataframe and scale it using the same training scaler
            new_team = pd.DataFrame([[quarter, dept, day, team, target, smv, wip, overtime,
                                      incentive, idle_time, idle_men, style_changes, workers]],
                                    columns=X.columns)
            new_team_scaled = scaler.transform(new_team)

            # Predict using the RSB model
            pred = svm_rbf.predict(new_team_scaled)[0]

            print("\n-----------------------------------------")
            if pred == 1:
                print("PREDICTION: This team is likely to MEET or EXCEED their productivity target.")
            else:
                print("PREDICTION: This team is at risk of FAILING to meet their productivity target.")
            print("-----------------------------------------")

            run_again = input("Would you like to test another team setup? (yes/no): ")
            if run_again.lower() != 'yes':
                break
        except ValueError:
            print("Invalid input. Please ensure you are typing numerical values.")

predict_productivity()

For this analysis, we were tasked with predicting whether factory teams in a garment manufacturing business could successfully meet their assigned daily productivity targets. We selected a business dataset tracking roughly 1,200 shifts, which included variables such as overtime minutes, the specific day of the week, financial incentives, and the total number of workers assigned to the floor. To prepare the data, we cleaned up incomplete records (such as empty Work-In-Progress columns for the finishing department) and converted our text data into numerical codes.

We deployed a Support Vector Machine utilizing an RBF (Radial Basis Function) kernel. Unlike a Linear kernel that draws a straight line to separate data, the RBF kernel evaluates the multi-dimensional distance between data points, creating flexible, bubble-like boundaries around complex patterns. Because it relies heavily on distance, scaling our data was an absolutely mandatory step so that massive metrics like 7,000 overtime minutes didn't mathematically drown out highly important, smaller variables like the 0.80 target score. The model achieved a solid accuracy of 74%, with an impressive recall of roughly 90%, meaning it is highly effective at identifying the conditions where teams succeed. For management, this interactive model serves as a valuable resource scheduling tool: before a shift starts, managers can plug in the target goals, the available workforce, and overtime budgets to see if the team is statistically set up for success, or if resources need to be shifted to avoid a failed quota.